# Day 4 — Sharpe Ratio & Sortino Ratio (daily) + Ranking

Compute:

- **Sharpe Ratio**: `(Rp − Rf) / Std(Rp) * sqrt(252)`
- **Sortino Ratio**: `(Rp − Rf) / downside_std * sqrt(252)`

Where **Rf = 6.5%** (annual) used as RBI repo rate proxy, converted to daily for daily-return-based calculations.

Ranking is produced across all funds.

In [2]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px

In [3]:
# --- Repo-root detection ---

_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()


def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'daily_returns_all_schemes.csv').exists() and (cand / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'daily_returns_all_schemes.csv').exists() and (parent / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent


_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

returns_path = DATA_DIR / 'daily_returns_all_schemes.csv'
fund_path = DATA_DIR / 'fund_master_clean.csv'

if not returns_path.exists():
    raise FileNotFoundError(f'Missing file: {returns_path.resolve()}')
if not fund_path.exists():
    raise FileNotFoundError(f'Missing file: {fund_path.resolve()}')

print('DATA_DIR:', DATA_DIR.resolve())

DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [4]:
returns_df = pd.read_csv(returns_path)
fund_df = pd.read_csv(fund_path)

returns_df['date'] = pd.to_datetime(returns_df['date'], errors='coerce')
returns_df['amfi_code'] = pd.to_numeric(returns_df['amfi_code'], errors='coerce').astype('Int64')
returns_df['daily_return'] = pd.to_numeric(returns_df['daily_return'], errors='coerce')

fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')

# Merge name if missing
if 'scheme_name' not in returns_df.columns:
    returns_df = returns_df.merge(fund_df[['amfi_code','scheme_name']], on='amfi_code', how='left')

returns_df = returns_df.dropna(subset=['amfi_code','daily_return']).copy()

print('Daily returns rows:', len(returns_df))
print('Schemes:', returns_df['amfi_code'].nunique())

Daily returns rows: 64280
Schemes: 40


In [5]:
# --- Parameters ---

RF_ANNUAL = 0.065  # 6.5%
TRADING_DAYS = 252

# Convert annual risk-free to daily risk-free (simple approximation)
rf_daily = RF_ANNUAL / TRADING_DAYS

print('rf_daily:', rf_daily)

rf_daily: 0.00025793650793650796


In [6]:
def sharpe_ratio(r: pd.Series, rf_d: float) -> float:
    r = r.dropna().astype(float)
    if len(r) < 2:
        return np.nan
    mean_r = r.mean()
    std_r = r.std(ddof=1)
    if std_r == 0 or np.isnan(std_r):
        return np.nan
    return (mean_r - rf_d) / std_r * np.sqrt(TRADING_DAYS)


def sortino_ratio(r: pd.Series, rf_d: float) -> float:
    r = r.dropna().astype(float)
    if len(r) < 2:
        return np.nan
    mean_r = r.mean()
    # Downside deviation: only negative days relative to rf_d
    downside = r[r < rf_d] - rf_d
    if len(downside) < 2:
        return np.nan
    downside_std = downside.std(ddof=1)
    if downside_std == 0 or np.isnan(downside_std):
        return np.nan
    return (mean_r - rf_d) / downside_std * np.sqrt(TRADING_DAYS)


In [7]:
# --- Compute ratios per fund ---

ratios = (
    returns_df.groupby(['amfi_code','scheme_name'])['daily_return']
    .agg(sharpe_ratio=lambda s: sharpe_ratio(s, rf_daily),
         sortino_ratio=lambda s: sortino_ratio(s, rf_daily),
         mean_daily_return='mean',
         std_daily_return='std')
    .reset_index()
)

# Rank
ratios['sharpe_rank'] = ratios['sharpe_ratio'].rank(ascending=False, method='dense').astype('Int64')
ratios['sortino_rank'] = ratios['sortino_ratio'].rank(ascending=False, method='dense').astype('Int64')

print('Top 10 Sharpe:')
display(ratios.sort_values('sharpe_ratio', ascending=False).head(10))

print('Top 10 Sortino:')
display(ratios.sort_values('sortino_ratio', ascending=False).head(10))

Top 10 Sharpe:


,amfi_code,scheme_name,sharpe_ratio,sortino_ratio,mean_daily_return,std_daily_return,sharpe_rank,sortino_rank
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,1.068224,1.554821,0.000768,0.007575,1,1
30,120843,Kotak Flexicap Fund - Regular - Growth,0.965561,1.485905,0.000773,0.008475,2,2
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,0.919047,1.387028,0.000804,0.009427,3,3
25,120505,ICICI Pru Midcap Fund - Regular - Growth,0.883256,1.321088,0.000830,0.010288,4,4
19,119551,SBI Bluechip Fund - Regular Plan - Growth,0.860977,1.302473,0.000656,0.007330,5,5
38,149323,DSP Midcap Fund - Regular - Growth,0.832885,1.201805,0.000754,0.009464,6,6
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,0.808268,1.184424,0.000772,0.010097,7,7
9,118632,Nippon India Large Cap Fund - Regular - Growth,0.758851,1.121482,0.000619,0.007545,8,8
16,119094,Axis Midcap Fund - Regular - Growth,0.730547,1.091546,0.000734,0.010347,9,9
3,101206,ABSL Frontline Equity Fund - Regular - Growth,0.717409,1.078795,0.000609,0.007768,10,10


Top 10 Sortino:


,amfi_code,scheme_name,sharpe_ratio,sortino_ratio,mean_daily_return,std_daily_return,sharpe_rank,sortino_rank
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,1.068224,1.554821,0.000768,0.007575,1,1
30,120843,Kotak Flexicap Fund - Regular - Growth,0.965561,1.485905,0.000773,0.008475,2,2
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,0.919047,1.387028,0.000804,0.009427,3,3
25,120505,ICICI Pru Midcap Fund - Regular - Growth,0.883256,1.321088,0.000830,0.010288,4,4
19,119551,SBI Bluechip Fund - Regular Plan - Growth,0.860977,1.302473,0.000656,0.007330,5,5
38,149323,DSP Midcap Fund - Regular - Growth,0.832885,1.201805,0.000754,0.009464,6,6
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,0.808268,1.184424,0.000772,0.010097,7,7
9,118632,Nippon India Large Cap Fund - Regular - Growth,0.758851,1.121482,0.000619,0.007545,8,8
16,119094,Axis Midcap Fund - Regular - Growth,0.730547,1.091546,0.000734,0.010347,9,9
3,101206,ABSL Frontline Equity Fund - Regular - Growth,0.717409,1.078795,0.000609,0.007768,10,10


In [8]:
# --- Save ranked outputs ---

out_path = DATA_DIR / 'sharpe_sortino_ranked_rf6_5.csv'
ratios.to_csv(out_path, index=False)

print('Wrote:', out_path.resolve())

Wrote: C:\Mutual Fund Analytics\Data\processed\sharpe_sortino_ranked_rf6_5.csv
